# 04 — Consumo da API em homologação

Este notebook chama a API de previsão.

Ele funciona com API local ou com URL publicada no Railway.

Antes de executar as células, suba a API seguindo `02_api_bitcoin/README_API.md`.

Se a API estiver em `http://127.0.0.1:8000`, não precisa alterar nada.

Se a API estiver em `http://127.0.0.1:8001`, rode antes no PowerShell:

```powershell
$env:API_BASE_URL="http://127.0.0.1:8001"
```

Se a API estiver no Railway, rode antes no PowerShell:

```powershell
$env:API_BASE_URL="https://SUA_API.up.railway.app"
```

A chave precisa ser a mesma configurada em `API_KEY` ao subir a API.

Para aula local, o padrão é:

```powershell
$env:API_KEY="aula4-local-key"
```

In [ ]:
from pathlib import Path
import json
import os

import requests


def find_lesson_dir():
    current = Path.cwd()
    if current.name == "4_Ciclo_Vida_MLOps_Bitcoin":
        return current
    candidate = current / "4_Ciclo_Vida_MLOps_Bitcoin"
    if candidate.exists():
        return candidate
    return current


LESSON_DIR = find_lesson_dir()
RUNTIME_DIR = LESSON_DIR / "runtime"
ARTIFACTS_DIR = RUNTIME_DIR / "artifacts"
API_BASE_URL = os.getenv("API_BASE_URL", "http://127.0.0.1:8000")
API_KEY = os.getenv("API_KEY", "aula4-local-key")

print("API:", API_BASE_URL)

## Carregar uma amostra

O notebook de treino salva `runtime/artifacts/sample_payload.json`.

Use essa amostra para testar o contrato da API.

In [ ]:
sample_path = ARTIFACTS_DIR / "sample_payload.json"
feature_path = ARTIFACTS_DIR / "feature_columns.json"

if sample_path.exists():
    payload = json.loads(sample_path.read_text(encoding="utf-8"))
    print("Amostra carregada:", sample_path)
elif feature_path.exists():
    feature_columns = json.loads(feature_path.read_text(encoding="utf-8"))["feature_columns"]
    payload = {"features": {column: 1.0 for column in feature_columns}}
    payload["features"]["price_usd_current"] = 65000.0
    print("Amostra sintética criada com todas as colunas de feature_columns.json.")
else:
    payload = None
    print("Amostra não encontrada. Execute o notebook 01 antes do teste completo.")

payload


## Testar `/health`

`/health` não exige chave.

In [ ]:
try:
    response = requests.get(f"{API_BASE_URL}/health", timeout=10)
    print(response.status_code)
    print(response.json())
except requests.RequestException as exc:
    print("API indisponível. Suba a API local ou configure API_BASE_URL para o Railway.")
    print(exc)

## Testar `/predict`

`/predict` exige o cabeçalho `X-API-Key`.

Se receber `401`, a chave enviada no notebook é diferente da variável `API_KEY` configurada na API.

Se receber erro de conexão, a API não está rodando ou `API_BASE_URL` aponta para a porta errada.

In [ ]:
headers = {"X-API-Key": API_KEY}

if payload is None:
    print("Sem payload válido. Execute o notebook 01 antes de chamar /predict.")
else:
    try:
        response = requests.post(f"{API_BASE_URL}/predict", headers=headers, json=payload, timeout=20)
        print("Status:", response.status_code)
        print(response.json())
    except requests.RequestException as exc:
        print("Não foi possível chamar a API.")
        print("Verifique se a API está rodando e se API_BASE_URL está correto.")
        print(exc)


## Critérios de homologação

Considere aprovado quando:

- `/health` responder;
- `/predict` rejeitar requisições sem chave;
- `/predict` aceitar uma linha válida com chave;
- o retorno indicar o alias correto;
- o resultado estiver coerente com a escala do preço.